<a href="https://colab.research.google.com/github/AeroCardia-DS/aerocardia/blob/main/aerocardia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Business Contex

## Project Goals
Analyze testing datasets to identify correlations and opportunities for system improvements.
Provide data-driven recommendations to optimize operations and reduce costs.
Enhance participants’ skills in statistical analysis, data modeling, and communicating results to non-technical audiences.
Suggest outcomes from our DOE tests to increase system consistency

## Project Outcomes
Clear identification of process inefficiencies and performance bottlenecks.
Actionable recommendations to improve operational efficiency and productivity.
Stakeholder-ready reports and visualizations that drive informed decision-making.

## Project Deliverables

Cleaned and analyzed operational dataset with key performance metrics.
Statistical analysis report highlighting inefficiencies and root causes.
Visualizations and dashboards illustrating findings.
Final presentation with prioritized recommendations for process improvements.

## 8-Week Timeline (50 Hours per Participant)

- Week 1 – Project Kickoff & Data Understanding (5 hrs)
Define project scope, KPIs, and success criteria.
Review datasets, data dictionaries, and context for testing.
- Week 2 – Data Cleaning & Integration (6 hrs)
Clean and preprocess testing data.
Integrate multiple data sources as needed (e.g., testing logs).
- Week 3 – Exploratory Data Analysis (6 hrs)
Conduct statistical analysis to identify trends, anomalies, and bottlenecks.
Share initial findings for feedback.
- Week 4 – Metric Development & Hypothesis Testing (6 hrs)
Define key testing metrics (e.g., quality, consistency).
Test hypotheses to validate root causes of inefficiencies.
- Week 5 – Visualization & Reporting (7 hrs)
Develop visualizations (dashboards, charts) to represent findings.
Draft interim report with preliminary recommendations.
- Week 6 – Recommendation Development (6 hrs)
Refine insights and translate analysis into actionable recommendations.
Prioritize recommendations based on impact and feasibility.
- Week 7 – Final Report & Documentation (7 hrs)
Compile final report including methodology, analysis, and recommendations.
Prepare supporting documentation for long-term use.
- Week 8 – Presentation & Handover (7 hrs)
Present findings and recommendations to stakeholders.
Deliver final report, visualizations, and documentation package.

### **Data Dictionary:**

The data dictionary is **the planned list of features**, so it is just listing what the **feature name** is, **what it means**, the
**theoretical equation for calculating it**, and what **type of metric it is**.

- Pulmonary Alveolar Ventilation (VA): Tidal Volume minus dead space and respiratory rate
- Cardiac Baroreflex sensitivity (BRS): Coupling of HRV and beat-to-beat pressure estimates
- Pulmonary Breathing Efficiency: From Tidal Volume and respiratory rate you can estimate overall breathing efficiency. Can factor in different humidities
- Pulmonary Breathing Frequency and Regularity: Timing and regularity of breaths. Irregular can indicate disorders such as Cheyne-Stokes or apneas
- Cardiac Cardiac Output (CO): HR and estimated SV can be used to calculate cardiac output
- Other Center of mass & stability: Acceleration and tilt data
- Other Core Body Temperature (estimate): Using ambient and exhaled air temperature. Monitoring for fever, heat stroke, or hypothermia. Humidity can help with this estimate. Humidity can relate this to heat stress and dehydration risk
- Pulmonary Dead Space Ventilation (VD): Estimate dead space using tidal volume and CO2 kinetics
- Other Fluid Balance: nan
- Cardiac Heart Rate: PPG waveform
- Cardiac Heart Rate Variability (HRV): Indicator of stress levels, autonomic balance, and overall cardiovascular health
- Other Hydration and Metabolic Stress Correlation: HR, breath humidity, and CO2
- Other Hydration Status V: Track evaporative losses and link to respiratory patterns and heart rate
- Pulmonary? Inhaled air composition: Ambient or specific humidity
- Pulmonary Inspiratory & Expiratory Flow Rate: Airflow limitations or respiratory diseases
- Pulmonary Lung compliance & elastance: From pressure volume dynamics
- Other Metabolic Rate: RER?
- Pulmonary Oxygen Delivery (DO2): CO and arterial oxygen amount (closely tied to SPO2 and O2)
- Other Physical Activity Level (PAL): Acceleration to estimate (sedentary, light, moderate, or vigorous activity)
- Other Posture and Gait Analysis: Tilt and acceleration to assess posture, balance and gait. For neurological or musculoskeletal health
- Pulmonary Pulmonary Vascular Resistance (PVR): Indirect estimate from CO2 HR and SPO2. Potential insights for pulmonary hypertension or cardiovascular stress
- Pulmonary Respiratory Exchange Ratio (RER): Ratio of CO2 production to O2 consumption (VCO2/VO2) provides insights into substrate being used for energy production (carbs vs fats) and metabolic rate
- Pulmonary Respiratory muscle strength/endurance indices: Inspiratory/expiratory pressures and fatigue patterns
- Pulmonary Respiratory Rate: Breaths per minute
- Other Respiratory water loss: Difference of humidity in vs out
- Other Resting Metabolic Rate (RMR): VO2 at rest, and using age gender and weight you estimate baseline energy expenditure
- Cardiac Stroke Volume (SV): Can be inferred from heart rate, SPO2 and CO2. Changes in HR and SPO2 can correlate with changes in SV
- Other Sympathovagal Balance: HRV, Respiratory rate, and CO2 levels to determine balance between fight or flight and rest and digest states
- Other Thermoregulatory response: nan
- nan VCO2: Exhaled CO₂, VT, RR
- Pulmonary Ventilation-perfusion (V/Q) Ratio: Deviation can be linked to pulmonary embolism or chronic lung diseases
- Pulmonary Ventilatory Efficiency: Effectiveness of gas exchange
- Pulmonary Ventilatory Equivalents (VE/VO2): Ventilation and estimated oxygen consumption
- Pulmonary VO2: Oxygen uptake
- Pulmonary VO2max: VO2 and heart rate data, or VO2 and VCO2 (RER)
- Pulmonary Work of breathing: Pressure-flow loop integration



# Import libraries

In [22]:
from unittest.mock import inplace

import mlflow
import pandas as pd
from mlflow.exceptions import MlflowException
import os
import numpy as np
import matplotlib as mpl

In [23]:
#!mlflow server

## Data
### Data Dictionary


In [72]:
col_name = ['Feature_Name', 'Description', 'Theorical_Calculation', 'Type']
data_dict = pd.read_excel('data/AeroCardia Data Dictionary Basic.xlsx', names=col_name)
data_dict.head(5)

,Feature_Name,Description,Theorical_Calculation,Type
0,Alveolar Ventilation (VA),Tidal Volume minus dead space and respiratory ...,V˙A=(VT−VD)×RR\dot V_A = (V_T - V_D)\times RR,Pulmonary
1,Baroreflex sensitivity (BRS),Coupling of HRV and beat-to-beat pressure esti...,ΔHR/ΔP\Delta HR / \Delta P slope,Cardiac
2,Breathing Efficiency,From Tidal Volume and respiratory rate you can...,Efficiency = VO₂ / Ventilation,Pulmonary
3,Breathing Frequency and Regularity,Timing and regularity of breaths. Irregular ca...,Frequency analysis of pressure waveform,Pulmonary
4,Cardiac Output (CO),HR and estimated SV can be used to calculate c...,CO=SV×HRCO = SV \times HR,Cardiac


In [79]:
data = data_dict[['Feature_Name', 'Description', 'Type']]
for entry in data.itertuples():
    print(f"- {entry.Type} {entry.Feature_Name}: {entry.Description}")



- Pulmonary Alveolar Ventilation (VA): Tidal Volume minus dead space and respiratory rate
- Cardiac Baroreflex sensitivity (BRS): Coupling of HRV and beat-to-beat pressure estimates
- Pulmonary Breathing Efficiency: From Tidal Volume and respiratory rate you can estimate overall breathing efficiency. Can factor in different humidities
- Pulmonary Breathing Frequency and Regularity: Timing and regularity of breaths. Irregular can indicate disorders such as Cheyne-Stokes or apneas
- Cardiac Cardiac Output (CO): HR and estimated SV can be used to calculate cardiac output
- Other Center of mass & stability: Acceleration and tilt data
- Other Core Body Temperature (estimate): Using ambient and exhaled air temperature. Monitoring for fever, heat stroke, or hypothermia. Humidity can help with this estimate. Humidity can relate this to heat stress and dehydration risk
- Pulmonary Dead Space Ventilation (VD): Estimate dead space using tidal volume and CO2 kinetics
- Other Fluid Balance: nan
- C

### DataSet - BaseLine

In [5]:
df1 = pd.read_csv('../data/Riipen TCC Data 1.csv')
df_baseline = df1.copy()

In [6]:
df_baseline.head()

,Elapsed Time (ms),Timestamp,Ambient Temperature,Ambient Humidity,Heart Rate (bpm),SpO2 (%),HRV SDNN (ms),Lung Volume (L),O2 (%),CO2 (%),...,Temperature BMP C (°C),IMU X (m/s²),IMU Y (m/s²),IMU Z (m/s²),BMI Gyro X (°/s),BMI Gyro Y (°/s),BMI Gyro Z (°/s),PPG IR,PPG Red,Battery State (%)
0,0,2026-02-02T17:27:41.274Z,28.076,47.459,NaN,NaN,NaN,4.5,20.226,0,...,32.387,0.139,-0.307,0.898,230,45,-86,105625,85832,0
1,91,2026-02-02T17:27:41.365Z,28.129,47.459,NaN,NaN,NaN,4.5,20.226,0,...,32.383,0.150,-0.303,0.923,-26,-155,-7,110983,88298,0
2,227,2026-02-02T17:27:41.501Z,28.129,53.525,NaN,NaN,NaN,4.5,20.226,0,...,32.391,0.144,-0.333,0.923,117,-16,-163,110983,88298,0
3,317,2026-02-02T17:27:41.591Z,28.183,53.525,NaN,NaN,NaN,4.5,20.226,0,...,32.390,0.144,-0.315,0.924,-54,-98,158,111385,88470,0
4,414,2026-02-02T17:27:41.688Z,28.161,58.163,NaN,NaN,NaN,4.5,20.226,0,...,32.392,0.153,-0.292,0.919,8,-72,-17,111385,88470,0


In [10]:
df_baseline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Elapsed Time (ms)            360 non-null    int64  
 1   Timestamp                    360 non-null    object 
 2   Ambient Temperature          360 non-null    float64
 3   Ambient Humidity             360 non-null    float64
 4   Heart Rate (bpm)             330 non-null    float64
 5   SpO2 (%)                     330 non-null    float64
 6   HRV SDNN (ms)                0 non-null      float64
 7   Lung Volume (L)              360 non-null    float64
 8   O2 (%)                       360 non-null    float64
 9   CO2 (%)                      360 non-null    int64  
 10  RER (ΔCO₂/ΔO₂)               360 non-null    float64
 11  Pressure BMP A (Pa)          360 non-null    float64
 12  Pressure BMP B (Pa)          360 non-null    float64
 13  Pressure BMP C Ambie

In [17]:
# Timestamp

# Create two columns
df_baseline['Date'] = df_baseline['Timestamp'].map(lambda x : x.split('T')[0])
df_baseline['Time'] = df_baseline['Timestamp'].map(lambda x : x.split('T')[1])

df_baseline['Date'] = pd.to_datetime(df_baseline['Date'])
df_baseline.head()

,Elapsed Time (ms),Timestamp,Ambient Temperature,Ambient Humidity,Heart Rate (bpm),SpO2 (%),HRV SDNN (ms),Lung Volume (L),O2 (%),CO2 (%),...,IMU Y (m/s²),IMU Z (m/s²),BMI Gyro X (°/s),BMI Gyro Y (°/s),BMI Gyro Z (°/s),PPG IR,PPG Red,Battery State (%),Date,Time
0,0,2026-02-02T17:27:41.274Z,28.076,47.459,NaN,NaN,NaN,4.5,20.226,0,...,-0.307,0.898,230,45,-86,105625,85832,0,2026-02-02,17:27:41.274Z
1,91,2026-02-02T17:27:41.365Z,28.129,47.459,NaN,NaN,NaN,4.5,20.226,0,...,-0.303,0.923,-26,-155,-7,110983,88298,0,2026-02-02,17:27:41.365Z
2,227,2026-02-02T17:27:41.501Z,28.129,53.525,NaN,NaN,NaN,4.5,20.226,0,...,-0.333,0.923,117,-16,-163,110983,88298,0,2026-02-02,17:27:41.501Z
3,317,2026-02-02T17:27:41.591Z,28.183,53.525,NaN,NaN,NaN,4.5,20.226,0,...,-0.315,0.924,-54,-98,158,111385,88470,0,2026-02-02,17:27:41.591Z
4,414,2026-02-02T17:27:41.688Z,28.161,58.163,NaN,NaN,NaN,4.5,20.226,0,...,-0.292,0.919,8,-72,-17,111385,88470,0,2026-02-02,17:27:41.688Z


In [21]:
missing_summary = df_baseline.isna().sum().sort_values(ascending=False)
missing_pct = missing_summary / len(df_baseline)
missing_pct

HRV SDNN (ms)                  1.000000
Heart Rate (bpm)               0.083333
SpO2 (%)                       0.083333
Elapsed Time (ms)              0.000000
Temperature BMP B (°C)         0.000000
Date                           0.000000
Battery State (%)              0.000000
PPG Red                        0.000000
PPG IR                         0.000000
BMI Gyro Z (°/s)               0.000000
BMI Gyro Y (°/s)               0.000000
BMI Gyro X (°/s)               0.000000
IMU Z (m/s²)                   0.000000
IMU Y (m/s²)                   0.000000
IMU X (m/s²)                   0.000000
Temperature BMP C (°C)         0.000000
Pressure Diff B-C (Pa)         0.000000
Temperature BMP A (°C)         0.000000
Timestamp                      0.000000
Pressure Diff A-C (Pa)         0.000000
Pressure BMP C Ambient (Pa)    0.000000
Pressure BMP B (Pa)            0.000000
Pressure BMP A (Pa)            0.000000
RER (ΔCO₂/ΔO₂)                 0.000000
CO2 (%)                        0.000000


## Observations:

* Rows: 360
* Columns: 28
* dtypes: float64(19), int64(8), object(1)